In [4]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, ReLU, Add, MaxPooling2D, Flatten, Dense, Input
)

# ==========================================
# LOAD AUGMENTED CSV
# ==========================================
name = "sasch"
df = pd.read_csv(
    fr"C:\Users\{name}\OneDrive\Desktop\iivp-2026-challenge\train_augmented.csv"
)

# ==========================================
# AUGMENTED DATASET DIRECTORY
# ==========================================

train_dir = fr"C:\Users\sasch\OneDrive\Desktop\iivp-2026-challenge\train_augmented"

X = []
y = []

# ==========================================
# LOAD IMAGES
# ==========================================

for _, row in df.iterrows():

    img_path = os.path.join(
        train_dir,
        str(row["Category"]),
        str(row["Id"]) + ".png"
    )

    img = Image.open(img_path).convert("L")

    # normalize
    img = np.array(img) / 255.0

    X.append(img)
    y.append(row["Category"])

# ==========================================
# CONVERT TO NUMPY
# ==========================================

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

# ==========================================
# RESHAPE FOR ResNet-Architecture
# ==========================================

X = X.reshape(-1, 32, 32, 1)

print("CNN shape:", X.shape)

# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
def residual_block(x, filters):
    shortcut = x
    x = Conv2D(filters, (3,3), padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Add()([x,shortcut])
    x = ReLU()(x)

    return x

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================

inputs = Input(shape=(32,32,1))

x = Conv2D(32,(3,3), padding="same")(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)

x = residual_block(x, 32)
x = residual_block(x, 32)

x = MaxPooling2D()(x)

x = Conv2D(64,(3,3), padding="same")(x)
x = BatchNormalization()(x)
x = ReLU()(x)

x = residual_block(x, 64)
x = residual_block(x, 64)

x = MaxPooling2D()(x)
x = Flatten()(x)
x = Dense(128, activation="relu")(x)
outputs = Dense(10, activation="softmax")(x)

model = Model(inputs, outputs)

# ==========================================
# Residual Blocks for ResNet-Architecture
# ==========================================
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# ==========================================
# TRAIN
# ==========================================

history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val)
)

# ==========================================
# VALIDATION ACCURACY
# ==========================================

val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)

# ==========================================
# LOAD TEST SET
# ==========================================

X_test = []
names = []

test_dir = fr"C:\Users\{name}\OneDrive\Desktop\iivp-2026-challenge\test\test"

for file in sorted(os.listdir(test_dir)):

    img_path = os.path.join(test_dir, file)

    img = Image.open(img_path).convert("L")

    img = np.array(img) / 255.0

    X_test.append(img)
    names.append(file)

X_test = np.array(X_test)

# reshape for CNN
X_test = X_test.reshape(-1, 32, 32, 1)

# ==========================================
# PREDICTIONS
# ==========================================

preds = model.predict(X_test)

pred_labels = np.argmax(preds, axis=1)

# ==========================================
# SUBMISSION CSV
# ==========================================

submission = pd.DataFrame({
    "Id": names,
    "Category": pred_labels
})

submission["Id"] = submission["Id"].str.replace(
    ".png",
    "",
    regex=False
)

submission.to_csv("submission_cnn.csv", index=False)

print("submission_cnn.csv saved")


Dataset shape: (34000, 32, 32)
CNN shape: (34000, 32, 32, 1)
Epoch 1/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 79s 89ms/step - accuracy: 0.9261 - loss: 0.2934 - val_accuracy: 0.8940 - val_loss: 0.4278
Epoch 2/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 76s 89ms/step - accuracy: 0.9820 - loss: 0.0616 - val_accuracy: 0.9706 - val_loss: 0.1095
Epoch 3/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 75s 88ms/step - accuracy: 0.9862 - loss: 0.0448 - val_accuracy: 0.9903 - val_loss: 0.0413
Epoch 4/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 75s 88ms/step - accuracy: 0.9910 - loss: 0.0308 - val_accuracy: 0.9653 - val_loss: 0.1693
Epoch 5/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 76s 89ms/step - accuracy: 0.9921 - loss: 0.0265 - val_accuracy: 0.9862 - val_loss: 0.0572
Epoch 6/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 77s 91ms/step - accuracy: 0.9925 - loss: 0.0256 - val_accuracy: 0.9547 - val_loss: 0.2237
Epoch 7/20
850/850 ━━━━━━━━━━━━━━━━━━━━ 78s 91ms/step - accuracy: 0.9956 - loss: 0.0166 - val_accuracy: 0.9834 - val_loss: 0.0886
Epoch 8/20
850/850 ━━━━━━━━━━